# Fast-FoundationStereo depth service

Serves stereo depth from NVlabs Fast-FoundationStereo as a resident HTTP service (`control/vla_bridge/fastfs_server.py`), through an ngrok HTTP tunnel. `run_vla_episode.py --depth-endpoint` and `depth_camera.DepthCameraClient` call it.

| | |
|---|---|
| Lane | HTTP depth service (isolated FastFS venv) |
| Port → tunnel | 3000 → ngrok HTTP |
| GPU runtime | GPU runtime; the former combined notebook ran it on A100 |
| Gated HF licences | none |
| Sign-offs before running | none |
| Default checkpoint | `NVlabs Fast-FoundationStereo 23-36-37 (gdown)` |
| Server log | `/content/fastfs_server.log` |
| Embedded files (written to `/content`) | `colab_server_utils.py`, `fastfs_server.py` |

Run top to bottom on a fresh runtime. Works from the VS Code Colab extension: nothing to upload.
Every file this notebook runs is written into `/content` by a cell below and checked against
its sha256. Run the teardown cell at the end of every session. The index of all notebooks,
with the local smoke and episode commands, is `control/vla_bridge/notebooks/README.md`.

Generated by `scripts/build_colab_notebooks.py` from `scripts/colab_notebook_specs.py` +
`control/vla_bridge` sources, so edit those and regenerate, not this file.

In [ ]:
import os

# Test 7 UAT crash investigation (Phase 12, 2026-09-27): Colab sessions that
# load multiple ML libraries (torch, opencv-headless, Fast-FoundationStereo's
# own deps) commonly end up with more than one bundled OpenMP runtime
# (libgomp vs libiomp) loaded into the same process -- the classic
# "OMP: Error #15" failure mode, which manifests as an abrupt, tracebackless
# kernel death the first time a *new* library's compute path runs for the
# first time (matches "crashes right as inference starts"). Set before any
# heavy imports so it applies process-wide to every cell below.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")
print("OpenMP duplicate-runtime guards set.")

**pyngrok legitimacy gate:** the next cell installs `pyngrok`, flagged
`SUS` in `11-RESEARCH.md`'s Package Legitimacy Audit. Do **not** run that
install cell until a human has completed the verification in this plan's
Task 3 `checkpoint:human-verify` (visit pypi.org/project/pyngrok, confirm
the `ngrok`/`alexdlaird` maintainer namespace and that `8.1.2` is a real
published version).

In [ ]:
# Colab notebook cell — DO NOT RUN until the human has completed the
# pyngrok legitimacy check in this plan's Task 3 checkpoint:human-verify
!pip install pyngrok==8.1.2

In [ ]:
# One-time token setup: HF Hub (for authenticated, faster/rate-limit-free checkpoint
# downloads) + ngrok (for the TCP tunnel). Checks for saved token files on Google Drive
# first (same MyDrive/SoARM-Research/.hf_token / .ngrok_token convention as other
# cells in this project), else falls back to a one-time getpass() prompt so tokens
# are never echoed or left in cell output/notebook history.
import getpass
from pathlib import Path

_hf_token = None
_ngrok_token = None
try:
    from google.colab import drive
    drive.mount("/content/drive")
    _hf_token_path = Path("/content/drive/MyDrive/SoARM-Research/.hf_token")
    _ngrok_token_path = Path("/content/drive/MyDrive/SoARM-Research/.ngrok_token")
    if _hf_token_path.exists():
        _hf_token = _hf_token_path.read_text().strip()
        print(f"HF token loaded from {_hf_token_path}")
    if _ngrok_token_path.exists():
        _ngrok_token = _ngrok_token_path.read_text().strip()
        print(f"ngrok token loaded from {_ngrok_token_path}")
except Exception as _exc:
    print(f"Drive mount/token read skipped ({type(_exc).__name__}: {_exc})")

if not _hf_token:
    _hf_token = getpass.getpass("HF Hub token (paste here): ")
if not _ngrok_token:
    _ngrok_token = getpass.getpass("ngrok authtoken (paste here): ")

from huggingface_hub import login
login(token=_hf_token)
print("Logged into HF Hub.")

from pyngrok import ngrok
ngrok.set_auth_token(_ngrok_token)
print("ngrok authtoken set.")

del _hf_token, _ngrok_token

## Server helper

The next cell writes `control/vla_bridge/colab_server_utils.py` into `/content/colab_server_utils.py`; the cell after it checks its sha256. The launch, diagnostic and teardown cells import it to stop stale servers, prove this notebook's server owns its port, and report logs, processes and GPU memory.

In [ ]:
%%writefile /content/colab_server_utils.py
"""Colab-side process/port helper for the per-backend policy-server notebooks.

Every notebook in control/vla_bridge/notebooks/ writes this file to
/content/colab_server_utils.py with a %%writefile cell (byte-identical to this file, checked
by a sha256 cell and by control/test_colab_notebooks.py) and imports it with
sys.path.insert(0, '/content'). Local control code never imports it.

What it guarantees for a launch cell:
- stop_servers() clears any earlier server: every process whose cmdline contains a known
  server marker, plus whatever is listening on the notebook's port.
- start_server() starts the server unbuffered (PYTHONUNBUFFERED=1) with its log in a file.
- wait_until_ready() only succeeds when the process this cell started (or one of its
  children) owns the port. A stale server that answers on the port is an error, not a
  success. It also fails on an early exit and, for patched launchers, on a missing banner.

The diagnostic helpers (log_report, process_report, gpu_report, health_report) print what a
partner needs to debug a run, and stop_server() is the teardown.

Imports: stdlib + psutil only (psutil is preinstalled on Colab).
"""

import json
import os
import re
import socket
import subprocess
import time
import urllib.request
from pathlib import Path
from typing import Callable, Mapping, Optional, Sequence

import psutil

SERVER_MARKERS = (
    "lerobot.async_inference.policy_server",
    "diffusion_policy_server.py",
    "groot_n17_server.py",
    "fastfs_server.py",
    "openvla_oft_server.py",
    "openvla_server.py",
    "octo_server.py",
    "wallx_server.py",
    "rdt1b_server.py",
    "groot_n15_server.py",
)

# Case-sensitive on purpose: "Error" covers KeyError, NotImplementedError, GatedRepoError and
# "Error in StreamActions"; lowercase prose such as "error budget" is not a match.
ERROR_PATTERN = re.compile(
    r"ERROR|Error|Traceback|Exception|OutOfMemory|out of memory|Killed|beginAllocateToPool"
)

_ACCESS_ERRORS = (psutil.AccessDenied, psutil.NoSuchProcess, psutil.ZombieProcess)


def _cmdline(proc) -> str:
    try:
        return " ".join(proc.cmdline())
    except _ACCESS_ERRORS:
        return ""
    except OSError:
        return ""


def _describe(pid: int) -> str:
    try:
        return f"PID {pid} ({_cmdline(psutil.Process(pid)) or '?'})"
    except _ACCESS_ERRORS:
        return f"PID {pid} (gone)"


def _tcp_connections(proc):
    # Colab's psutil may predate 6.0, where net_connections() replaced connections().
    method = getattr(proc, "net_connections", None) or proc.connections
    return method(kind="tcp")


def _read_lines(log_path) -> list:
    path = Path(log_path)
    if not path.exists():
        return []
    return path.read_text(errors="replace").splitlines()


def _tail(log_path, lines: int = 40) -> str:
    return "\n".join(_read_lines(log_path)[-lines:])


def listening_pids(port: int) -> set:
    """PIDs of every visible process with a TCP socket in LISTEN state on `port`."""
    pids = set()
    for proc in psutil.process_iter():
        try:
            for conn in _tcp_connections(proc):
                if conn.status == psutil.CONN_LISTEN and conn.laddr and conn.laddr.port == port:
                    pids.add(proc.pid)
        except _ACCESS_ERRORS:
            continue
        except OSError:
            continue
    return pids


def stop_servers(port: int, patterns: Sequence[str] = (), timeout_s: float = 15.0) -> list:
    """Stop every process matching a pattern plus the port's listener; return what was stopped."""
    me = os.getpid()
    targets = {}
    for proc in psutil.process_iter():
        if proc.pid == me:
            continue
        cmdline = _cmdline(proc)
        if cmdline and any(pattern and pattern in cmdline for pattern in patterns):
            targets[proc.pid] = proc
    for pid in listening_pids(port):
        if pid != me and pid not in targets:
            try:
                targets[pid] = psutil.Process(pid)
            except _ACCESS_ERRORS:
                continue
    stopped = []
    for pid, proc in sorted(targets.items()):
        description = f"PID {pid} ({_cmdline(proc) or '?'})"
        try:
            proc.terminate()
            stopped.append(description)
        except psutil.NoSuchProcess:
            continue
        except psutil.AccessDenied:
            print(f"Cannot stop {description}: access denied")
    _, alive = psutil.wait_procs(list(targets.values()), timeout=5)
    for proc in alive:
        try:
            proc.kill()
        except psutil.NoSuchProcess:
            pass
    psutil.wait_procs(alive, timeout=5)
    for line in stopped:
        print(f"Stopped {line}")
    deadline = time.monotonic() + timeout_s
    while True:
        holders = listening_pids(port) - {me}
        if not holders:
            return stopped
        if time.monotonic() >= deadline:
            raise RuntimeError(
                f"port {port} is still held by "
                + ", ".join(_describe(pid) for pid in sorted(holders))
                + f" after {timeout_s:.0f} s; stop it by hand before launching"
            )
        time.sleep(0.2)


def start_server(argv: Sequence[str], log_path,
                 env_overrides: Optional[Mapping[str, str]] = None,
                 cwd=None) -> subprocess.Popen:
    """Start `argv` unbuffered with stdout+stderr in `log_path` (truncated)."""
    env = os.environ.copy()
    env.update(env_overrides or {})
    env["PYTHONUNBUFFERED"] = "1"
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log = open(log_path, "wb", buffering=0)
    try:
        proc = subprocess.Popen(list(argv), stdout=log, stderr=subprocess.STDOUT, env=env,
                                cwd=None if cwd is None else str(cwd))
    finally:
        log.close()
    print(f"Started PID {proc.pid}: {' '.join(argv)}")
    print(f"Log → {log_path}")
    return proc


def _probe(port: int, health_url: Optional[str], ready: Optional[Callable[[dict], bool]]):
    """Return (answered, health): answered is True once the server counts as ready."""
    if health_url:
        try:
            with urllib.request.urlopen(health_url, timeout=2) as response:
                health = json.loads(response.read().decode())
        except Exception:
            return False, None
        if not isinstance(health, dict):
            return False, None
        ok = ready(health) if ready is not None else health.get("status") == "ready"
        return bool(ok), health
    try:
        with socket.create_connection(("127.0.0.1", port), timeout=1):
            return True, None
    except OSError:
        return False, None


def _own_pids(proc) -> set:
    pids = {proc.pid}
    try:
        pids.update(child.pid for child in psutil.Process(proc.pid).children(recursive=True))
    except _ACCESS_ERRORS:
        pass
    return pids


def wait_until_ready(proc, port: int, log_path, timeout_s: float,
                     health_url: Optional[str] = None,
                     ready: Optional[Callable[[dict], bool]] = None,
                     banner: Optional[str] = None,
                     settle_s: float = 2.0, poll_s: float = 1.0):
    """Wait until the server this cell started owns `port` and is ready.

    Returns the /health dict in health mode, None in TCP mode. Raises RuntimeError on an
    early exit, a foreign/stale owner or a missing launcher banner, and TimeoutError (after
    terminating `proc`) when the deadline passes.
    """
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        code = proc.poll()
        if code is not None:
            raise RuntimeError(
                f"server PID {proc.pid} exited with code {code} before it was ready. "
                f"Last log lines of {log_path}:\n{_tail(log_path)}"
            )
        answered, health = _probe(port, health_url, ready)
        if answered:
            owners = listening_pids(port)
            mine = _own_pids(proc)
            if not owners or not owners <= mine:
                foreign = ", ".join(_describe(pid) for pid in sorted(owners - mine))
                foreign = foreign or "no visible process"
                raise RuntimeError(
                    f"port {port} answered but is owned by {foreign}, not the server this cell "
                    f"started (PID {proc.pid}). That is a stale server; re-run the launch cell, "
                    "which stops old servers first."
                )
            time.sleep(settle_s)
            code = proc.poll()
            if code is not None:
                raise RuntimeError(
                    f"server PID {proc.pid} exited with code {code} right after binding port "
                    f"{port}. Last log lines of {log_path}:\n{_tail(log_path)}"
                )
            if banner is not None:
                text = Path(log_path).read_text(errors="replace") if Path(log_path).exists() else ""
                if banner not in text:
                    raise RuntimeError(
                        f"expected launcher banner '{banner}' not found in {log_path}, so this "
                        "is not the patched launcher. Re-run the embed cells, then the launch cell."
                    )
            print(f"READY: PID {proc.pid} ({_cmdline(psutil.Process(proc.pid))}) owns port "
                  f"{port}; log → {log_path}")
            return health
        time.sleep(poll_s)
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
    raise TimeoutError(
        f"server PID {proc.pid} was not ready on port {port} within {timeout_s:.0f} s "
        f"(terminated). Last log lines of {log_path}:\n{_tail(log_path)}"
    )


def log_report(log_path, tail: int = 80, max_matches: int = 40) -> None:
    """Print the last `tail` log lines and up to `max_matches` numbered error lines."""
    path = Path(log_path)
    if not path.exists():
        print(f"{path} does not exist (has the launch cell run on this runtime?)")
        return
    lines = _read_lines(path)
    print(f"--- last {tail} lines of {path} ({len(lines)} lines total) ---")
    for line in lines[-tail:]:
        print(line)
    matches = [(number, line) for number, line in enumerate(lines, start=1)
               if ERROR_PATTERN.search(line)]
    print(f"--- {len(matches)} lines matching ERROR_PATTERN (showing up to {max_matches}) ---")
    for number, line in matches[:max_matches]:
        print(f"{number}: {line}")
    if not matches:
        print("(none)")


def process_report(port: int, expected_marker: str, proc=None) -> None:
    """Print every known server process, then check the right server owns `port`."""
    print("--- running server processes ---")
    found = False
    now = time.time()
    for candidate in psutil.process_iter():
        cmdline = _cmdline(candidate)
        if not cmdline or not any(marker in cmdline for marker in SERVER_MARKERS):
            continue
        try:
            elapsed = now - candidate.create_time()
            rss_mb = candidate.memory_info().rss / 1e6
        except _ACCESS_ERRORS:
            continue
        found = True
        print(f"PID {candidate.pid}  up {elapsed:7.0f} s  RSS {rss_mb:8.1f} MB  {cmdline}")
    if not found:
        print("(no process with a known server marker)")
    owners = listening_pids(port)
    if not owners:
        raise RuntimeError(f"nothing is listening on port {port}; run the launch cell")
    for pid in sorted(owners):
        try:
            cmdline = _cmdline(psutil.Process(pid))
        except _ACCESS_ERRORS:
            cmdline = ""
        if expected_marker not in cmdline:
            raise RuntimeError(
                f"WRONG server on port {port}: PID {pid} ({cmdline or '?'}) does not contain "
                f"'{expected_marker}'. Run the teardown cell, then the launch cell."
            )
    if proc is not None:
        mine = _own_pids(proc)
        if not owners <= mine:
            raise RuntimeError(
                f"port {port} is owned by "
                + ", ".join(_describe(pid) for pid in sorted(owners))
                + f", not by the server this notebook started (PID {proc.pid})"
            )
    print(f"OK: {expected_marker} (PID {min(owners)}) owns port {port}")


def gpu_report() -> None:
    """Print GPU memory/utilisation and the processes using the GPU."""
    queries = (
        ["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
         "--format=csv"],
        ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv"],
    )
    for argv in queries:
        try:
            result = subprocess.run(argv, capture_output=True, text=True, timeout=30)
        except FileNotFoundError:
            print("no nvidia-smi (CPU runtime?): pick a GPU runtime for this notebook")
            return
        print(result.stdout.rstrip() or result.stderr.rstrip())


def health_report(port: int) -> dict:
    """GET /health on the local server, pretty-print it and return it."""
    url = f"http://127.0.0.1:{port}/health"
    try:
        with urllib.request.urlopen(url, timeout=10) as response:
            health = json.loads(response.read().decode())
    except Exception as exc:
        print(f"GET {url} failed: {type(exc).__name__}: {exc}")
        raise
    print(json.dumps(health, indent=2, sort_keys=True))
    return health


def stop_server(proc, port: int, patterns: Sequence[str] = ()) -> None:
    """Teardown: stop `proc` (if alive) and any leftover server, then confirm the port is free."""
    if proc is not None and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=10)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=10)
        print(f"Stopped PID {proc.pid}")
    stop_servers(port, patterns)
    print(f"port {port} is free")


In [ ]:
import hashlib
from pathlib import Path

EXPECTED_SHA256 = '2d8f63cbfdf5e679c268dd4f4436129e8fa8f30acfd5c5f5d88eb1082c2bf2a8'
actual = hashlib.sha256(Path('/content/colab_server_utils.py').read_bytes()).hexdigest()
if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f'/content/colab_server_utils.py has sha256 {actual[:12]}, expected {EXPECTED_SHA256[:12]}: '
        're-run the %%writefile cell above. To change the file, edit control/vla_bridge/colab_server_utils.py '
        'and regenerate the notebooks (scripts/build_colab_notebooks.py).'
    )
print(f'/content/colab_server_utils.py matches control/vla_bridge/colab_server_utils.py (sha256 {actual[:12]})')

## Install and load Fast-FoundationStereo

FastFS runs in its own virtualenv (`/content/fastfs_venv`, torch 2.6.0 cu124), so it never
reinstalls torch/cuDNN under a running PolicyServer.

In [ ]:
import subprocess
from pathlib import Path
FASTFS_DIR = Path('/content/Fast-FoundationStereo')
if not (FASTFS_DIR / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/NVlabs/Fast-FoundationStereo', str(FASTFS_DIR)], check=True)
print('FastFS source revision:', subprocess.check_output(['git', '-C', str(FASTFS_DIR), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
# Separate filesystem environment: never reinstall torch/cuDNN under a running PolicyServer.
import sys
import subprocess
from pathlib import Path
FASTFS_VENV = '/content/fastfs_venv'
FASTFS_PYTHON = f'{FASTFS_VENV}/bin/python'
if 'depth_server_process' in globals() and depth_server_process.poll() is None:
    raise RuntimeError('Stop the depth service before changing its environment; use teardown below.')
if not Path(FASTFS_PYTHON).exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'virtualenv'], check=True)
    subprocess.run([sys.executable, '-m', 'virtualenv', FASTFS_VENV], check=True)
subprocess.run([FASTFS_PYTHON, '-m', 'pip', 'install', 'torch==2.6.0', 'torchvision==0.21.0', 'xformers',
                '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)
subprocess.run([FASTFS_PYTHON, '-m', 'pip', 'install', '-r', '/content/Fast-FoundationStereo/requirements.txt'], check=True)
subprocess.run([FASTFS_PYTHON, '-m', 'pip', 'install', 'flask'], check=True)
# No GUI patch: the resident adapter calls the model directly, never run_demo.py.

Use the trusted NVlabs `23-36-37` research checkpoint below. The launch cell loads it once
in the isolated environment, warms inference, then serves subsequent requests in
process. No CLI subprocess or model reload occurs per depth request.

Download the checkpoint and configuration into the isolated service’s weights directory:

In [ ]:
import os

os.makedirs("/content/Fast-FoundationStereo/weights/23-36-37", exist_ok=True)

!pip install -q gdown
!gdown "1W1V1H64l9bAi97boEQQ2ueNzzGmSMz-E" \
    -O /content/Fast-FoundationStereo/weights/23-36-37/model_best_bp2_serialize.pth
!gdown "1GDBRYL-ZaLpXEtWfGFRJvkBc_2sywjgj" \
    -O /content/Fast-FoundationStereo/weights/23-36-37/cfg.yaml

print(os.listdir("/content/Fast-FoundationStereo/weights/23-36-37"))

## Start the resident depth service
Run the cells below. The first writes the versioned adapter (embedded so this
notebook is standalone) and the second checks its sha256. The launch cell starts it as a
separate OS process using FastFS’s virtualenv and waits for warmup. Keep the model resident
for the entire episode. `/health` only becomes available after warmup. Logs survive notebook
errors at `/content/fastfs_server.log`.

In [ ]:
%%writefile /content/fastfs_server.py
"""Resident FastFS HTTP service. Run with the isolated FastFS Python interpreter.

Adapter follows NVlabs/Fast-FoundationStereo scripts/run_demo.py's model API;
no demo subprocess, GUI, point-cloud construction, or intermediate files.
"""
import argparse
import base64
import hashlib
import io
import threading
import time
from pathlib import Path

import cv2
import numpy as np


class FastFSModel:
    def __init__(self, checkpoint, valid_iters=8, max_disp=192):
        import torch
        from core.utils.utils import InputPadder
        from Utils import AMP_DTYPE

        self.torch, self.padder, self.amp_dtype = torch, InputPadder, AMP_DTYPE
        if not torch.cuda.is_available():
            raise RuntimeError("FastFS requires CUDA in its isolated environment")
        checkpoint = Path(checkpoint).resolve()
        digest = hashlib.sha256()
        with checkpoint.open('rb') as f:
            for block in iter(lambda: f.read(1024 * 1024), b''):
                digest.update(block)
        # Research checkpoints serialize the complete model. Only load the
        # operator-selected, trusted NVlabs checkpoint, never request data.
        self.model = torch.load(checkpoint, map_location='cpu', weights_only=False)
        self.model.args.valid_iters = valid_iters
        self.model.args.max_disp = max_disp
        self.model.cuda().eval()
        self.valid_iters = valid_iters
        self.metadata = dict(checkpoint=checkpoint.name, checkpoint_sha256=digest.hexdigest(),
                             valid_iters=valid_iters, max_disp=max_disp, units='metres')

    def __call__(self, left_bgr, right_bgr, intrinsics, baseline):
        torch = self.torch
        tensors = [torch.from_numpy(cv2.cvtColor(x, cv2.COLOR_BGR2RGB)).to('cuda', dtype=torch.float32)
                   .permute(2, 0, 1).unsqueeze(0) for x in (left_bgr, right_bgr)]
        padding = self.padder(tensors[0].shape, divis_by=32, force_square=False)
        left, right = padding.pad(*tensors)
        with torch.inference_mode(), torch.amp.autocast('cuda', dtype=self.amp_dtype):
            disparity = self.model.forward(left, right, iters=self.valid_iters,
                                           test_mode=True, optimize_build_volume='pytorch1')
        disparity = padding.unpad(disparity.float()).cpu().numpy().reshape(left_bgr.shape[:2])
        depth = np.full(disparity.shape, np.nan, dtype=np.float32)
        valid = np.isfinite(disparity) & (disparity > 0)
        np.divide(float(intrinsics[0]) * baseline, disparity, out=depth, where=valid)
        return depth


def create_app(infer):
    from flask import Flask, jsonify, request
    app = Flask(__name__)
    app.config['MAX_CONTENT_LENGTH'] = 20 * 1024 * 1024
    inference_lock = threading.Lock()

    @app.get('/health')
    def health():
        return jsonify(status='ready', protocol_version=1, model=infer.metadata)

    @app.post('/depth')
    def depth():
        if not inference_lock.acquire(blocking=False):
            return jsonify(error='depth service busy'), 429
        try:
            body = request.get_json()
            frames = []
            for key in ('left_png_b64', 'right_png_b64'):
                raw = base64.b64decode(body[key], validate=True)
                frame = cv2.imdecode(np.frombuffer(raw, dtype=np.uint8), cv2.IMREAD_COLOR)
                if frame is None:
                    raise ValueError('invalid PNG')
                frames.append(frame)
            if frames[0].shape != frames[1].shape or max(frames[0].shape[:2]) > 992:
                raise ValueError('expected equal stereo images with dimensions <=992')
            K = np.asarray(body['intrinsics_flat'], dtype=float)
            baseline = float(body['baseline_m'])
            if K.shape != (9,) or not np.isfinite(K).all() or K[0] <= 0 or not np.isfinite(baseline) or baseline <= 0:
                raise ValueError('invalid calibration')
            started = time.perf_counter()
            result = infer(*frames, K.tolist(), baseline)
            elapsed = (time.perf_counter() - started) * 1000
            buf = io.BytesIO()
            np.save(buf, np.asarray(result, dtype=np.float32), allow_pickle=False)
            return jsonify(depth_npy_b64=base64.b64encode(buf.getvalue()).decode('ascii'),
                           model=infer.metadata, inference_ms=elapsed,
                           request_id=body.get('request_id'))
        except (ValueError, KeyError, TypeError) as exc:
            return jsonify(error=str(exc)), 400
        finally:
            inference_lock.release()
    return app


def main():
    import sys
    parser = argparse.ArgumentParser()
    parser.add_argument('--repo', required=True)
    parser.add_argument('--checkpoint', required=True)
    parser.add_argument('--port', type=int, default=3000)
    parser.add_argument('--warmup-width', type=int, default=800)
    parser.add_argument('--warmup-height', type=int, default=576)
    args = parser.parse_args()
    sys.path.insert(0, str(Path(args.repo).resolve()))
    model = FastFSModel(args.checkpoint)
    # Compilation happens before the socket opens and readiness is reported.
    warmup = np.zeros((args.warmup_height, args.warmup_width, 3), dtype=np.uint8)
    model(warmup, warmup, [800., 0., 400., 0., 800., 288., 0., 0., 1.], .05)
    print('FastFS model loaded and warmed; ready', flush=True)
    create_app(model).run(host='0.0.0.0', port=args.port, threaded=True, use_reloader=False)


if __name__ == '__main__':
    main()


In [ ]:
import hashlib
from pathlib import Path

EXPECTED_SHA256 = '2be4cf85970c3a47419a55822fe43ab79c1cfbaccb4fe13c045bddb0f225f59b'
actual = hashlib.sha256(Path('/content/fastfs_server.py').read_bytes()).hexdigest()
if actual != EXPECTED_SHA256:
    raise RuntimeError(
        f'/content/fastfs_server.py has sha256 {actual[:12]}, expected {EXPECTED_SHA256[:12]}: '
        're-run the %%writefile cell above. To change the file, edit control/vla_bridge/fastfs_server.py '
        'and regenerate the notebooks (scripts/build_colab_notebooks.py).'
    )
print(f'/content/fastfs_server.py matches control/vla_bridge/fastfs_server.py (sha256 {actual[:12]})')

## Launch

The launch cell first stops any earlier server: every process whose command line contains one of `fastfs_server.py`, and whatever listens on port 3000.
It then starts this notebook's server as a separate OS process (unbuffered, log at `/content/fastfs_server.log`).
It only succeeds once **its own** still-running process owns port 3000.
A stale server answering on the port is an error, never a successful launch (in the 2026-10-18 live session a stale stock server made a diffusion launch look successful).

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, '/content')
from colab_server_utils import start_server, stop_servers, wait_until_ready

PORT = 3000
DEPTH_SERVER_LOG = '/content/fastfs_server.log'
STOP_PATTERNS = ['fastfs_server.py']
FASTFS_PYTHON = '/content/fastfs_venv/bin/python'
FASTFS_DIR = '/content/Fast-FoundationStereo'
FASTFS_MODEL = f'{FASTFS_DIR}/weights/23-36-37/model_best_bp2_serialize.pth'

if not Path(FASTFS_MODEL).is_file():
    raise FileNotFoundError(f'Checkpoint missing: {FASTFS_MODEL}; run the install and checkpoint download cells first')

stop_servers(PORT, STOP_PATTERNS)
depth_server_process = start_server(
    [FASTFS_PYTHON, '-u', '/content/fastfs_server.py', '--repo', FASTFS_DIR,
     '--checkpoint', FASTFS_MODEL, '--port', '3000'],
    DEPTH_SERVER_LOG,
    env_overrides={'PYTHONPATH': FASTFS_DIR, 'KMP_DUPLICATE_LIB_OK': 'TRUE', 'OMP_NUM_THREADS': '1'},
    cwd=FASTFS_DIR,
)
print('Loading and warming FastFS once. First compilation may take several minutes.')
health = wait_until_ready(
    depth_server_process, PORT, DEPTH_SERVER_LOG,
    timeout_s=600, health_url='http://127.0.0.1:3000/health', ready=lambda h: h.get('status') == 'ready' and h.get('protocol_version') == 1,
)
print('Depth service ready:', health)

This exact request/response shape (`left_png_b64`/`right_png_b64`/
`intrinsics_flat`/`baseline_m` request, `depth_npy_b64` response) is what
`depth_camera.DepthCameraClient.compute_depth()` (Plan 12-06 Task 1) sends
and parses -- keep them in sync if either side changes.

## Open the depth tunnel

In [ ]:
from pyngrok import ngrok

# Port 3000 and a plain HTTP tunnel type -- separate from the gRPC notebooks' 5173 TCP
# tunnel, which specifically needs "tcp" for HTTP/2 semantics. This is a plain
# JSON-over-HTTP POST endpoint, so ngrok's default HTTP tunnel type works fine here.
depth_tunnel = ngrok.connect(3000, "http")
print(f"Depth endpoint reachable at: {depth_tunnel.public_url}/depth")
SMOKE = 'curl -s https://<depth_tunnel>/health'
EPISODE = '.venv/bin/python run_vla_episode.py --out outputs/depth_episode_UNIQUE_NAME --server-address POLICY_TCP_HOST:PORT --checkpoint victorvanhalst/smolvla_so101_cube --stereo-camera-index "CCB Camera" --depth-endpoint https://DEPTH_TUNNEL/depth --depth-calibration stereo_calibration.json --depth-every-n-steps 1 --max-steps 60 --instruction "Pick the red cube and place it in the bowl"'
for placeholder in ('https://<depth_tunnel>', 'https://DEPTH_TUNNEL'):
    SMOKE = SMOKE.replace(placeholder, depth_tunnel.public_url)
    EPISODE = EPISODE.replace(placeholder, depth_tunnel.public_url)
print('Local smoke (run in control/):')
print(SMOKE)
print('Local episode (run in control/):')
print(EPISODE)

Pass the printed `.../depth` URL as `depth_camera.py`'s `--endpoint` argument
(Plan 12-06 Task 1) or `run_vla_episode.py`'s `--depth-endpoint` flag
(Task 3).

## Run the local episode with depth enabled

Run this notebook through the depth tunnel cell, and start a SmolVLA policy server with
`grpc_smolvla.ipynb` (through its tunnel cell). Each notebook's kernel starts its own ngrok
agent. If ngrok refuses a second agent session for the same authtoken, run both notebooks on
one runtime (ports 3000 and 5173 do not collide) and open the second tunnel from this
notebook's kernel with `ngrok.connect(5173, "tcp")`. A teardown cell's `ngrok.kill()` closes
every tunnel of its kernel. Do not run teardown until the local process exits.
Use the updated local checkout containing the async recorder. In the local `control/` directory:

```bash
.venv/bin/python run_vla_episode.py --out outputs/depth_episode_UNIQUE_NAME \
  --server-address POLICY_TCP_HOST:PORT \
  --checkpoint victorvanhalst/smolvla_so101_cube \
  --stereo-camera-index "CCB Camera" \
  --depth-endpoint https://DEPTH_TUNNEL/depth \
  --depth-calibration stereo_calibration.json --depth-every-n-steps 1 \
  --max-steps 60 --instruction "Pick the red cube and place it in the bowl"
```

Use the actual addresses printed by the tunnel cells and a fresh output directory.
The depth health check runs before hardware opens. Each eligible step submits depth
unless a prior request is outstanding; busy skips are explicit, never a backlog.
Motion continues during requests. At shutdown the arm returns to start and the last
request is drained before writing the final depth summary.

Inspect `depth_summary.json`: `complete` must be positive; review `failed` and
`skipped_busy`. `episode.jsonl` links each submitted request to
`depth_overhead/STEP.json`, which records the final success/failure and depth file.
`depth.jsonl` indexes these completed results. `camera_depth_aligned_left/` and
`camera_depth_aligned_right/` are the exact rectified/resized depth inputs;
`depth_calibration.json` preserves calibration. Depth is float32 metres with NaN
for invalid pixels. `action_details.waypoint_sends` records send outcomes;
`raw_model_output` retains the policy output before validation. Driver sends are
not proof of measured physical arrival. This fixes recording; it does not add a
depth input feature to the existing RGB-only SmolVLA checkpoint.

A successful test requires saved valid depth throughout the episode AND observed
smooth motion. Check the manifest timings and the physical run; a normal step-count
termination by itself does not establish success.

Quick reachability check from the laptop (the tunnel cell prints it with the URL filled in):

```bash
curl -s https://<depth_tunnel>/health
```

## Diagnostics

Run any of these after the launch cell, also from a restarted kernel: each cell re-declares
what it needs.
- Log: the tail of `/content/fastfs_server.log`, then every line matching `ERROR`, `Error`, `Traceback`,
  `Exception`, `OutOfMemory`, `out of memory`, `Killed` or `beginAllocateToPool`.
- Processes: every known server process on this runtime (stale servers from other notebooks
  show up here); fails unless `fastfs_server.py` owns port 3000.
- Package versions, GPU memory, and `/health`.

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import log_report

log_report('/content/fastfs_server.log')

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import process_report

process_report(3000, 'fastfs_server.py', globals().get('depth_server_process'))

In [ ]:
FASTFS_PYTHON = '/content/fastfs_venv/bin/python'
!{FASTFS_PYTHON} -c "import torch; print('cuda available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"
!{FASTFS_PYTHON} -c "import torch, torchvision; print('torch', torch.__version__, '| torchvision', torchvision.__version__)"

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import gpu_report

gpu_report()

In [ ]:
import sys

sys.path.insert(0, '/content')
from colab_server_utils import health_report

health_report(3000)

## Teardown (every session, no exceptions)

An unattended open tunnel into a Colab-hosted server is an
elevation-of-privilege risk (T-11-08 in this plan's threat register) —
stop both processes at the end of every session, not just when finished
for the day: the cell below disconnects this notebook's tunnel, stops the
ngrok agent, stops the server and confirms port 3000 is free. Run it only
after the local episode or smoke run has finished.

In [ ]:
# Run ONLY when the episode has finished and local depth results have drained.
import sys

sys.path.insert(0, '/content')
from colab_server_utils import stop_server
from pyngrok import ngrok

if 'depth_tunnel' in globals():
    try:
        ngrok.disconnect(depth_tunnel.public_url)
        print(f'Disconnected {depth_tunnel.public_url}')
    except Exception as exc:
        print(f'Tunnel already closed ({type(exc).__name__}: {exc})')
ngrok.kill()
print('ngrok agent stopped.')
stop_server(globals().get('depth_server_process'), 3000, ['fastfs_server.py'])